# JUnit Behavior Tests

CSC-239 · Module 7 · Lesson 4 of 4

You have checked printed output and handled expected failures. Now you will write automated behavior checks with JUnit, run them in IJava, and confirm that a useful test can detect a relevant mistake.

Select the **Java** kernel in your Workspace. Start with a fresh kernel and run cells in order. This lesson includes a visible JUnit dependency setup cell. Run it before the worked example and rerun it after restarting the kernel. Its first download requires access to Maven Central.


## Learning Goals

- Write JUnit Jupiter tests for normal, boundary and exceptional behavior.
- Run selected notebook-defined tests and interpret passing and deliberately failing results.


## Why This Matters

A label-count method may change as an application grows. Repeatable tests help you notice if a change breaks normal results, boundary behavior, or input validation.


## Check Your Starting Point

Explain an input/result contract, a public instance method, @Override, and a specific exception handler. Trace try-with-resources and explain why a test for an expected exception must also detect the case where no exception occurs.

**My explanation:**


## Concept

### Add the testing library

An **external dependency** is a library added beyond the Java standard library. JUnit supplies test annotations, assertions, and a runner. An import gives a class a short name in source; it does not download the library that contains that class.

This lesson uses JUnit Jupiter 5.13.4 and JUnit Platform 1.13.4 from one pinned standalone JAR. A JAR is a Java archive file containing compiled classes and related resources. The word standalone here means that this archive includes the components needed for our runner.

IJava provides a special setup command beginning with %maven. That line tells the kernel to obtain a library and add it to the current kernel's classpath, the locations Java searches for classes. The coordinate after it identifies the library's group, artifact, and version, separated by colons. It is a kernel setup command, not a Java statement to paste into a .java file.

The setup cell appears below the concept reading. Run it once in each fresh kernel. It normally prints nothing. If the first download fails, resolve the network or dependency error before continuing; missing-library errors in later cells are not failures of the method you are testing.

### Check one small behavior

A **unit test** is an automated check of a small behavior using controlled inputs and an expected result. Begin with a contract: for nonnegative group and kit counts, the label count is their product; negative counts are rejected.

An **@Test method** is a JUnit Jupiter method marked for discovery and execution as a test. Like @Override, @Test is an annotation written before a declaration. Their meanings differ: @Override asks the compiler to check an inherited method relationship; @Test tells JUnit which test method to run.

Our test methods are public, nonstatic, return void, and take no parameters. Each creates its own inputs through the calls it makes. A test should not depend on another test running first.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
class SumTest {
    public SumTest() { }
    @Test
    public void addsTwoCounts() {
        Assertions.assertEquals(5, 2 + 3);
    }
}
```

This defines a test class; defining it does not run the test. The explicit empty constructor follows the class pattern you already learned. The annotation marks addsTwoCounts for the runner.

### Make the result check precise

A **test assertion** checks actual behavior against an expected result and reports failure when they differ. Assertions.assertEquals takes the expected value first and the actual value second. Use the problem's stated behavior to choose the expected value.

```java
import org.junit.jupiter.api.Assertions;
int actual = 2 * 3;
Assertions.assertEquals(6, actual);
System.out.println("Check passed");
```

This prints Check passed. A successful assertion normally prints nothing itself. An incorrect expected value such as 7 would make this assertion fail before the following print. Inside a JUnit test, the runner records that assertion failure as a failed test.

Do not calculate the expected value by copying the same implementation expression used by the method. That can repeat the same mistake on both sides. For a small known example, choose the expected result independently.

### Test a failure as behavior

An expected application exception can be a successful test outcome. Call the method inside try. If it returns normally, Assertions.fail reports that the expected exception did not occur. Catch only the exception required by the application contract and check its message when that message is part of the contract.

In the worked example, negativeGroups calls labelCount with a negative group count. It then calls Assertions.fail if no exception was raised. Its catch handles IllegalArgumentException, the application failure we expect. A JUnit assertion failure is a different kind of failure; this catch does not swallow it.

Keeping Assertions.fail after the method call closes an easy testing gap. A test that merely catches an exception and does nothing else may pass even if the method incorrectly returns without throwing.

### Select and run a test class

A **test runner** discovers tests, executes them, and collects results. In a conventional project, an IDE or build tool often starts it. In this notebook, a provided Launcher scaffold starts it explicitly.

A **class literal**, such as `LabelToolsTest.class`, refers to the class itself. It is not a String containing the class name and does not construct a test object. We pass it to DiscoverySelectors.selectClass so JUnit can select the current notebook-defined class.

Read the runner as a series of familiar method calls:

1. LauncherDiscoveryRequestBuilder.request starts a request. selectors adds our selected class, and build produces the completed request. The chained calls use each returned object for the next operation.
2. SummaryGeneratingListener collects the result counts. Here a listener is an object the runner notifies as testing progresses; you do not need to implement this supplied listener.
3. LauncherFactory.openSession opens a runner session. Try-with-resources closes it after the tests finish.
4. getLauncher provides the launcher. registerTestExecutionListeners connects the summary collector. execute runs the request.
5. getSummary supplies the collected results. We print the succeeded and failed test counts.

The scaffold is included with every complete solution so it does not depend on a runner object left over from another cell. After editing a test class, rerun its definition and the runner. When in doubt, restart the kernel, run the dependency setup, and execute in order.

A normal kernel completion is not the same thing as a green test suite. The Launcher can return normally while its summary reports Failed: 1. Read the test counts. Zero failures is useful only when the intended tests were actually discovered and executed; our baseline also requires exactly three successes.

### Verify the test can detect a mistake

A **regression test cycle** keeps a behavior check, confirms that it fails for a relevant mistake, and reruns it after a repair. A regression is behavior that used to work and becomes incorrect after a change.

First run the correct suite. Then deliberately change one known expected value to an incorrect value and run again. The lesson's three-test suite should change from three successes and no failures to two successes and one failure. Restore the correct expected value and confirm three successes again.

This controlled change checks that the test is being discovered and its assertion matters. It does not prove that every possible defect is covered. The independent task asks you to add distinct boundary and invalid-input cases, then account for the increased test count.


### Prepare this kernel

Run the next cell to add the pinned JUnit archive to this IJava kernel. The first run downloads it from Maven Central; later runs may reuse the workspace’s download cache. No output normally means setup completed. If it reports an error, fix that setup error before running the Java tests. Every new kernel needs this setup even when the archive is already cached. The video uses the same pinned archive on a conventional Java classpath, as described in its transcript.


In [ ]:
%maven org.junit.platform:junit-platform-console-standalone:1.13.4


## Video Demonstration

Identify the normal, zero, and invalid-input tests. Predict the succeeded and failed counts before the suite runs. The video uses conventional Java source; the notebook setup command stays in IJava.

<video controls preload="metadata" width="960">
  <source src="media/04_junit_behavior_tests/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/04_junit_behavior_tests/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the junit behavior tests demonstration transcript](media/04_junit_behavior_tests/transcript.md).


## Worked Example

**Subgoal 1: state the production behavior.** LabelTools multiplies nonnegative counts and rejects a negative input.

**Subgoal 2: check three distinct behaviors.** LabelToolsTest checks a normal product, a zero boundary, and the required exception.

**Subgoal 3: run and count.** Select LabelToolsTest.class, execute the request, and read both result counts.


In [ ]:
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());


Expected output:

```text
Succeeded: 3
Failed: 0
```

All three checks agree with the method contract, so JUnit reports three succeeded and zero failed. The negative-input test succeeds because the required exception occurs with the expected message. The method is not expected to return a numeric value for that invalid input.


## Predict, Run, Trace, and Explain

### Predict the added boundary test

Read the complete program before running it. It adds zeroKits to the worked example. Count the @Test methods selected by LabelToolsTest.class. For each, predict whether its assertion or expected-exception check will succeed. Then predict both printed result counts. Explain why the expected application exception can count as a successful test. Record your prediction before running the next cell or opening the answer.

My count of selected @Test methods:

My predicted result of each method and why:

My predicted Succeeded and Failed lines:

Why the required exception can be a success:

Why the new boundary changes the expected executed count:


In [ ]:
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());


In a fresh Java kernel, first run the visible dependency setup cell, then run the complete Java cell. The imports give short names to classes; they do not install JUnit. Keep your prediction and compare both result counts with the actual output. Check that their sum equals the number of intended tests. Explain any mismatch and retain your post-run explanation. If you restart the kernel, rerun the dependency setup before the full Java program.

My original predicted counts:

My actual Succeeded and Failed lines:

My expected and actual total tests:

What the dependency cell does versus what imports do:

What I rerun after a kernel restart:

My corrected post-run explanation:

### Trace discovery, assertions and an exception-test gap

Complete the table for the four-test prediction program. For each method, identify its inputs, the independently chosen expectation and whether success means returning a number or raising the required exception. Point to the class literal that selects the test class. Trace the supplied runner through request creation, selection, listener registration, execution, summary reading and session closure. Explain why the external dependency setup, imports, test-class definition and runner execution perform different jobs. Then compare the two complete programs below to find a case that an incomplete exception test can miss.

| @Test method | Inputs | Expected behavior | Why it succeeds or fails |
|---|---|---|---|
| positiveCounts | | | |
| zeroGroups | | | |
| negativeGroups | | | |
| zeroKits | | | |


The class literal and what it selects:

The runner operations in order:

When the session closes relative to summary printing:

Dependency setup versus import versus definition versus execution:

Why these tests do not need a particular execution order:

My post-run explanation:

<details>
<summary>Show answer</summary>

The selected LabelToolsTest class has four @Test methods. positiveCounts checks the known product 2 times 3 against 6. zeroGroups checks that zero groups produce zero labels. negativeGroups succeeds because the application raises IllegalArgumentException with the required message; the line that would report a missing exception is skipped. zeroKits checks the separate zero-kit boundary with groups still positive. All four agree with the production contract, so the runner reports Succeeded: 4 and Failed: 0. The class literal selects the test class; defining the class alone would not execute its tests. The new listener receives this run's counts, and the runner session closes after execute finishes. The expected values come from the behavior contract, not from duplicating the implementation expression. The setup cell loads the pinned JUnit archive into the current kernel. Every full Java program supplies all imports, its application/test definitions and its own runner, but the external library still must be available in that kernel. A normally completed kernel cell does not by itself prove these test expectations passed; the printed counts provide the relevant evidence. The request selects LabelToolsTest.class. A new SummaryGeneratingListener collects results only after it is registered with the launcher. execute runs the request, and the try-with-resources statement closes the session before the two summary prints. The setup supplies JUnit classes; imports shorten their names; @Test marks the test methods; the explicit runner discovers and executes them. Each test makes its own calls with local inputs and does not rely on another test running first.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 4
Failed: 0
```

Common error: Counting the expected application exception as a failed test. Counting ordinary helper methods as discovered @Test methods. Reusing the worked example's result count after adding another test.

</details>


### Expose a false pass from an incomplete exception test

These two complete programs deliberately remove the production method's negative-input guard. Their multiplication remains unchanged. The first also removes Assertions.fail from negativeGroups; the second restores that one test line while keeping the production defect. Predict both summaries before running either program. Trace what negativeGroups does when labelCount(-1, 3) returns normally, and explain whether the catch executes. Then run both complete programs and compare the counts. Identify which version exposes the missing application exception and why a normal kernel completion is not enough to judge the tests. Finally run the complete restored four-test program, with both its original guard and full exception test, and explain why that passing result is stronger than the first program's false pass.

My predicted and actual incomplete-check counts:

What the faulty application call returns:

Whether the catch runs after that return:

My predicted and actual counts after restoring Assertions.fail:

Why the specific catch does not hide that assertion failure:

My actual restored four-test program counts:

Why the false pass and the repaired pass mean different things:

My post-run explanation:


**First program:**


In [ ]:
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());


**Comparison program:**


In [ ]:
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

Both supplied comparison programs deliberately remove the production guard, so labelCount(-1, 3) incorrectly returns -3. In the first, negativeGroups also lacks Assertions.fail. The application call returns normally, the catch is skipped, and the test reaches its end without reporting the missing exception. The three numeric tests still pass, so the summary falsely looks good at 4 successes and 0 failures. Restoring Assertions.fail while leaving the production defect in place makes negativeGroups report the missing exception; the summary becomes 3 successes and 1 failure. The catch handles only IllegalArgumentException and therefore does not hide the JUnit assertion failure. The Launcher still completes normally in both cases. To restore the application contract, put back the negative-input guard as well as retaining the complete exception test; the correct four-test program then reports 4 successes and 0 failures.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 4
Failed: 0
```

Common error: Treating the first zero-failure count as proof that invalid inputs are rejected. Removing Assertions.fail because a catch is already present. Repairing only the expected test count while leaving the production defect.

**Check case 2.** The production call returns when it should throw. Assertions.fail records a failed test, and the specific IllegalArgumentException catch does not intercept the assertion failure.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 3
Failed: 1
```

**Check case 3.** The negative-input call now raises the required application exception with the correct message, and the full test checks it. All four tests then agree with the stated contract.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 4
Failed: 0
```

</details>


## Guided Practice

Complete these tasks in order. The intentionally empty code cells are safe to run, but remain unfinished until you write and check your code.


### Complete the test and runner connections

The displayed draft is incomplete and for reading only. Copy it into the empty work cell. Replace TEST_MARK, CHECK_EQUAL, REPORT_MISSING_EXCEPTION and TEST_CLASS with `Test`, `assertEquals`, `fail` and `LabelToolsTest`, each once. Keep every other declaration and statement unchanged. The pinned dependency setup remains in its own cell. Predict both result counts, run the completed program, and explain each replacement's job. Explain why this answer has a different executed count from the four-test prediction program.

This sample is for repair:

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @TEST_MARK
    public void positiveCounts() {
        Assertions.CHECK_EQUAL(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.REPORT_MISSING_EXCEPTION("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(TEST_CLASS.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```


My four replacements and their jobs:

My predicted result counts:

My actual result counts:

Which value is expected and which is actual:

Why the expected executed count differs from the prediction suite:

My post-run explanation:

<details>
<summary>Show answer</summary>

TEST_MARK is Test, which marks positiveCounts for JUnit discovery. CHECK_EQUAL is assertEquals, which compares independently chosen expected 6 with the actual labelCount result. REPORT_MISSING_EXCEPTION is fail, which reports a test failure if the invalid-input call returns normally. TEST_CLASS is LabelToolsTest; the following .class forms the class literal passed to selectClass. The remaining two @Test methods and the runner stay unchanged. This is the exact three-test canonical program, so the completed answer reports 3 successes and 0 failures after dependency setup.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 3
Failed: 0
```

Common error: Replacing the class literal with a String. Reversing the expected and actual assertion roles. Leaving out the missing-exception check. Using the four-test count for a draft containing only three tests.

</details>


### Make one assertion fail and restore it

First run the intact four-test starter and record its counts. Change only the expected value in positiveCounts from 6 to 7. Keep the production method, all inputs, the other tests and runner unchanged. Predict and run this deliberately incorrect test program. Explain which test should fail and why the application's actual result remains 6. Notice that the Java cell can finish normally while JUnit reports the failure. Restore expected 6, rerun the whole program and check the complete four-test passing count. Keep your before, changed and restored observations.


In [ ]:
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());


My intact-suite counts:

My predicted counts with expected 7:

My actual counts with expected 7:

The failed test, expected value and actual value:

What normal kernel completion does and does not mean:

My restored-suite counts:

My post-run explanation:

<details>
<summary>Show answer</summary>

With expected 7, positiveCounts compares 7 against the correct actual result 6 and fails. The two zero-boundary tests and the required-exception test still succeed, so this four-test suite reports Succeeded: 3 and Failed: 1. The runner records the failed assertion and returns normally; its successful kernel completion does not turn the failed test into a pass. Restoring expected 6 returns the complete suite to 4 successes and 0 failures. This controlled edit checks that the selected test runs and its assertion can report a mismatch, while the later production-defect task checks the suite against a different kind of mistake.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(7, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 3
Failed: 1
```

Common error: Changing production multiplication to make an intentionally wrong expectation pass. Assuming one failure must always leave two successes, regardless of the suite size. Stopping after the failing run without restoring the correct expectation. Reading normal kernel completion as a successful JUnit suite.

**Additional test: `Restored expected value 6`.** All four assertions or required-exception checks agree with the unchanged application contract after the intentional expectation error is removed.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 4
Failed: 0
```

</details>


### Use the tests to repair the production method

The displayed program changes the production return expression from multiplication to addition. The four tests keep their correct expectations. Keep this faulty draft in Markdown. Predict the result of each test and both summary counts. Explain which tests detect the mistake and which still succeed. Copy the complete program into the empty work cell and repair only the production expression so it meets the stated product contract. Preserve every test and the runner. Run the repair and verify the full four-test passing count. Explain why changing the expected values would conceal the defect.

This sample is for repair:

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups + kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```


My predicted result of each faulty test:

My predicted faulty Succeeded and Failed counts:

The arithmetic results that violate the contract:

Why the negative-input test still succeeds:

My repaired expression:

My actual repaired counts:

Why I preserved the test expectations:

My post-run explanation:

<details>
<summary>Show answer</summary>

The selected LabelToolsTest class has four @Test methods. positiveCounts checks the known product 2 times 3 against 6. zeroGroups checks that zero groups produce zero labels. negativeGroups succeeds because the application raises IllegalArgumentException with the required message; the line that would report a missing exception is skipped. zeroKits checks the separate zero-kit boundary with groups still positive. All four agree with the production contract, so the runner reports Succeeded: 4 and Failed: 0. The class literal selects the test class; defining the class alone would not execute its tests. The new listener receives this run's counts, and the runner session closes after execute finishes. The expected values come from the behavior contract, not from duplicating the implementation expression. In the faulty version, addition returns 5 for (2, 3), 3 for (0, 3), and 2 for (2, 0). Those results violate the independently specified product expectations, so the three numeric tests fail. The guard still rejects a negative count with the correct message, so negativeGroups succeeds. The faulty summary is 1 success and 3 failures. Restoring groups * kits repairs all three numeric behaviors while preserving the rejection rule. The unchanged tests then report 4 successes and 0 failures.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 4
Failed: 0
```

Common error: Changing the tests to agree with addition even though the contract requires a product. Changing the correct negative-input guard while fixing arithmetic. Assuming a passing rejection test means the numeric behavior is also correct.

</details>


## Independent Practice

### Build the three seat-count behavior tests

Use the provided starter, which includes all imports, the unchanged SeatMath.remaining implementation, an explicit SeatMathTest constructor and the complete Launcher scaffold. Fill in three public, nonstatic, parameterless void @Test methods in SeatMathTest. Check capacity 8 with 3 reserved returns 5, and capacity 8 with 8 reserved returns 0. The third test calls remaining(8, 9), uses Assertions.fail if that call returns, and catches only IllegalArgumentException to check the exact message `Reservation must fit the capacity.` Keep the runner selecting `SeatMathTest.class` and keep the production method unchanged. Choose expectations from the contract and give each test its own inputs. Run the pinned dependency setup in each fresh kernel before the complete Java program. Predict both summary counts before running, then explain why the required exception represents a successful behavior check. In the next stage, deliberately change expected 5 to 6, restore it, and add the three required boundary tests.

**Provided starter code.** Keep this definition with your own implementation:

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class SeatMath {
    public static int remaining(int capacity, int reserved) {
        if (capacity < 0 || reserved < 0 || reserved > capacity) {
            throw new IllegalArgumentException("Reservation must fit the capacity.");
        }
        return capacity - reserved;
    }
}
class SeatMathTest {
    public SeatMathTest() { }
    // Add the three required test methods here.
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(SeatMathTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

My three test names and the behavior each checks:

My expected values chosen from the contract:

My predicted baseline summary:

My actual baseline summary:

How the invalid-input test detects a missing exception:

Which class the runner selects:

My post-run explanation:


### Verify the failure cycle and add missing boundaries

First use the complete three-test baseline. Change only the expected 5 in your test of capacity 8 with 3 reserved to 6, predict and record both actual summary lines, and explain the failed assertion even if the Java cell completes normally. Restore expected 5 and rerun before adding tests. Then follow the three additions below, predicting and recording the total executed count and both result counts after each step.

Add one public @Test method named negativeCapacity that calls SeatMath.remaining(-1, 0), uses Assertions.fail if no exception occurs, catches only IllegalArgumentException and checks the exact required message. Run the complete suite and record its increased count. Keep that test and add negativeReservations with SeatMath.remaining(8, -1), using the same failure/message pattern; run again. Then add zeroCapacityAndReservations, asserting expected 0 for SeatMath.remaining(0, 0), and run again. Preserve the original three tests, their correct expectations, the provided production code and the full runner. Explain what distinct behavior each added test preserves; these checks target the contract, not a particular line in its condition.

Repair any mismatch and rerun the affected suite. Retain the deliberate failure, restored baseline and all three expanded-suite observations with your own post-run explanation.

| Suite version | Predicted Succeeded/Failed | Actual Succeeded/Failed | Expected/actual total |
|---|---|---|---|
| Baseline three correct tests | | | |
| Baseline with expected 5 changed to 6 | | | |
| Restored baseline | | | |
| Add negativeCapacity | | | |
| Also add negativeReservations | | | |
| Also add zeroCapacityAndReservations | | | |


Why the wrong-expectation cell can complete normally while a test fails:

Why each added test preserves a distinct required behavior:

How each exception test detects an unexpected normal return:

My repairs and final six-test counts:

My post-run explanation:


<details>
<summary>Show answer</summary>

The provided SeatMath rejects negative capacity, negative reservations and reservations beyond capacity. someSeatsAvailable checks a normal remaining value of 5. allSeatsReserved checks the zero-remaining boundary. tooManyReservations requires IllegalArgumentException with the exact contract message; Assertions.fail makes an unexpected normal return fail the test instead of slipping through. Each test supplies its own input values. The runner selects SeatMathTest.class and reports 3 successes with 0 failures for the complete baseline. Its imports and runner are included, but the pinned dependency must first be loaded into each new kernel. Changing expected 5 to 6 makes only someSeatsAvailable fail, so the three-test suite reports 2 successes and 1 failure while the runner still returns normally. Restoring 5 produces 3 successes and 0 failures. Adding negativeCapacity gives four tests; adding negativeReservations gives five; adding zeroCapacityAndReservations gives six. All added expectations follow the fixed contract, so the successive correct summaries are 4/0, 5/0 and 6/0. The two negative-input tests check both the required exception and its message, including the case where no exception occurs. The zero/zero case confirms that a capacity of zero is valid when no seats are reserved. Keeping the earlier tests means the original behaviors remain covered as the suite grows.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class SeatMath {
    public static int remaining(int capacity, int reserved) {
        if (capacity < 0 || reserved < 0 || reserved > capacity) {
            throw new IllegalArgumentException("Reservation must fit the capacity.");
        }
        return capacity - reserved;
    }
}
class SeatMathTest {
    public SeatMathTest() { }
    @Test
    public void someSeatsAvailable() {
        Assertions.assertEquals(5, SeatMath.remaining(8, 3));
    }
    @Test
    public void allSeatsReserved() {
        Assertions.assertEquals(0, SeatMath.remaining(8, 8));
    }
    @Test
    public void tooManyReservations() {
        try {
            SeatMath.remaining(8, 9);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(SeatMathTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 3
Failed: 0
```

Common error: Editing the supplied production method to match a faulty test. Catching a broad failure instead of the required IllegalArgumentException. Leaving out Assertions.fail after the call expected to throw. Selecting the previous LabelToolsTest class instead of SeatMathTest. Counting zero failures as enough when the starter still has no completed tests.

**Additional test: Deliberately wrong baseline expectation: 5 becomes 6.** The actual normal result remains 5. Only that assertion fails, leaving two successes and one failure; this is an intended observed failure, not the final solution.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class SeatMath {
    public static int remaining(int capacity, int reserved) {
        if (capacity < 0 || reserved < 0 || reserved > capacity) {
            throw new IllegalArgumentException("Reservation must fit the capacity.");
        }
        return capacity - reserved;
    }
}
class SeatMathTest {
    public SeatMathTest() { }
    @Test
    public void someSeatsAvailable() {
        Assertions.assertEquals(6, SeatMath.remaining(8, 3));
    }
    @Test
    public void allSeatsReserved() {
        Assertions.assertEquals(0, SeatMath.remaining(8, 8));
    }
    @Test
    public void tooManyReservations() {
        try {
            SeatMath.remaining(8, 9);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(SeatMathTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 2
Failed: 1
```

**Additional test: Baseline plus negativeCapacity.** The new invalid-capacity test includes its own call, missing-exception safeguard and required message check. The original three tests remain, so four tests pass.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class SeatMath {
    public static int remaining(int capacity, int reserved) {
        if (capacity < 0 || reserved < 0 || reserved > capacity) {
            throw new IllegalArgumentException("Reservation must fit the capacity.");
        }
        return capacity - reserved;
    }
}
class SeatMathTest {
    public SeatMathTest() { }
    @Test
    public void someSeatsAvailable() {
        Assertions.assertEquals(5, SeatMath.remaining(8, 3));
    }
    @Test
    public void allSeatsReserved() {
        Assertions.assertEquals(0, SeatMath.remaining(8, 8));
    }
    @Test
    public void tooManyReservations() {
        try {
            SeatMath.remaining(8, 9);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
    @Test
    public void negativeCapacity() {
        try {
            SeatMath.remaining(-1, 0);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(SeatMathTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 4
Failed: 0
```

**Additional test: Also add negativeReservations.** The additional test checks a negative reservation count independently of test execution order. Keeping all earlier tests increases the passing count to five.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class SeatMath {
    public static int remaining(int capacity, int reserved) {
        if (capacity < 0 || reserved < 0 || reserved > capacity) {
            throw new IllegalArgumentException("Reservation must fit the capacity.");
        }
        return capacity - reserved;
    }
}
class SeatMathTest {
    public SeatMathTest() { }
    @Test
    public void someSeatsAvailable() {
        Assertions.assertEquals(5, SeatMath.remaining(8, 3));
    }
    @Test
    public void allSeatsReserved() {
        Assertions.assertEquals(0, SeatMath.remaining(8, 8));
    }
    @Test
    public void tooManyReservations() {
        try {
            SeatMath.remaining(8, 9);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
    @Test
    public void negativeCapacity() {
        try {
            SeatMath.remaining(-1, 0);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
    @Test
    public void negativeReservations() {
        try {
            SeatMath.remaining(8, -1);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(SeatMathTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 5
Failed: 0
```

**Additional test: Also add zeroCapacityAndReservations.** Zero capacity and zero reservations produce zero remaining seats. This sixth distinct test preserves valid boundary behavior alongside the three baseline and two negative-input tests.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class SeatMath {
    public static int remaining(int capacity, int reserved) {
        if (capacity < 0 || reserved < 0 || reserved > capacity) {
            throw new IllegalArgumentException("Reservation must fit the capacity.");
        }
        return capacity - reserved;
    }
}
class SeatMathTest {
    public SeatMathTest() { }
    @Test
    public void someSeatsAvailable() {
        Assertions.assertEquals(5, SeatMath.remaining(8, 3));
    }
    @Test
    public void allSeatsReserved() {
        Assertions.assertEquals(0, SeatMath.remaining(8, 8));
    }
    @Test
    public void tooManyReservations() {
        try {
            SeatMath.remaining(8, 9);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
    @Test
    public void negativeCapacity() {
        try {
            SeatMath.remaining(-1, 0);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
    @Test
    public void negativeReservations() {
        try {
            SeatMath.remaining(8, -1);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
    @Test
    public void zeroCapacityAndReservations() {
        Assertions.assertEquals(0, SeatMath.remaining(0, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(SeatMathTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 6
Failed: 0
```

</details>


## Summary

JUnit is an external dependency that must be loaded into each fresh kernel. @Test marks a method for the runner; assertions compare independently chosen expectations with actual behavior. A class literal selects the test class, and the runner reports executed test results. A controlled wrong expectation should produce a failure, and restoring it should return the suite to green.

Close the answers. Explain why an exception test needs Assertions.fail after a nonthrowing call and why zero failed tests is incomplete evidence without an expected executed count.


## Reflection

Choose a method from your own earlier practice. State one normal case, one boundary case, and one invalid-input case worth preserving as automated tests. Explain which relevant mistake each test could catch.

**My design and explanation:**

Next, you will use exceptions and automatic cleanup while reading and writing text files.


## Supplemental Reading

- [JUnit 5.13.4 assertions](https://docs.junit.org/5.13.4/user-guide/index.html#writing-tests-assertions) introduces Jupiter behavior checks.
- [JUnit Platform Launcher API](https://docs.junit.org/5.13.4/user-guide/index.html#launcher-api) explains test selection, execution, and listeners.
- [JUnit 5.13.4 Assertions API](https://docs.junit.org/5.13.4/api/org.junit.jupiter.api/org/junit/jupiter/api/Assertions.html) documents assertEquals and fail.
- [Original IJava dependency magics](https://github.com/SpencerPark/IJava/blob/master/docs/magics.md) explains kernel dependency setup.
